# 02 — Data Preprocessing & Leak-Free Pipeline Verification

**Project:** LoanWise AI  
**Role:** Member 1 (ML Model Development)  

This notebook demonstrates:
1. Separating the 11 raw input features (`Loan_ID` excluded) and `Loan_Status` (`Y -> 1`, `N -> 0`).
2. Performing stratified train/test splitting **before** fitting any imputer, scaler, or encoder (preventing data leakage).
3. Running `build_preprocessing_pipeline()` from `ml/src/preprocessing.py` and verifying zero remaining NaNs and clean transformed feature names.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "ml":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.src.data_loader import load_training_data
from ml.src.preprocessing import (
    build_preprocessing_pipeline,
    get_transformed_feature_names,
    prepare_features_and_target,
)

In [2]:
raw_df = load_training_data()
X, y = prepare_features_and_target(raw_df)

print("Prepared X shape:", X.shape)
print("Prepared y shape:", y.shape)
print("Feature columns:", X.columns.tolist())
print("Target distribution (1=Approved, 0=Rejected):")
print(y.value_counts())

Prepared X shape: (614, 11)
Prepared y shape: (614,)
Feature columns: ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area']
Target distribution (1=Approved, 0=Rejected):
Loan_Status
1    422
0    192
Name: count, dtype: int64


In [3]:
# Stratified split BEFORE fitting preprocessing (prevents data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

print("Train shape:", X_train.shape, "| Approval rate:", round(y_train.mean(), 4))
print("Test shape: ", X_test.shape, "| Approval rate:", round(y_test.mean(), 4))

Train shape: (491, 11) | Approval rate: 0.6864
Test shape:  (123, 11) | Approval rate: 0.6911


In [4]:
preprocessor = build_preprocessing_pipeline(scale_numeric=True)

# Fit ONLY on X_train, then transform both X_train and X_test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = get_transformed_feature_names(preprocessor)
print("Transformed train shape:", X_train_processed.shape)
print("Transformed test shape: ", X_test_processed.shape)
print("NaNs in processed train:", int(np.isnan(X_train_processed).sum()))
print("NaNs in processed test: ", int(np.isnan(X_test_processed).sum()))
print("\nTransformed feature names:")
for name in feature_names:
    print(" -", name)

Transformed train shape: (491, 18)
Transformed test shape:  (123, 18)
NaNs in processed train: 0
NaNs in processed test:  0

Transformed feature names:
 - num_continuous__Dependents
 - num_continuous__ApplicantIncome
 - num_continuous__CoapplicantIncome
 - num_continuous__LoanAmount
 - num_continuous__Loan_Amount_Term
 - num_continuous__TotalIncome
 - num_binary__Credit_History
 - categorical__Gender_Female
 - categorical__Gender_Male
 - categorical__Married_No
 - categorical__Married_Yes
 - categorical__Education_Graduate
 - categorical__Education_Not Graduate
 - categorical__Self_Employed_No
 - categorical__Self_Employed_Yes
 - categorical__Property_Area_Rural
 - categorical__Property_Area_Semiurban
 - categorical__Property_Area_Urban
